In [ ]:
# ==============================================================================
# Mã nguồn
# ĐỀ TÀI PRJ-04: PHÂN TÍCH BẢN CHẤT DỮ LIỆU BỒI THƯỜNG NGOẠI TRÚ (OUTPATIENT)
# File phân tích: C:\Users\Admin\Downloads\06_2521001108_PRJ04_Phan Thị Quế Trân (1).csv
# Nhóm 6: Nguyễn Ngọc Phương Thảo (2521001099) & Phan Thị Quế Trân (2521001108)
# ==============================================================================

import sys
import os
import re
import pandas as pd
import numpy as np

# Thiết lập UTF-8 hiển thị tiếng Việt trên Windows
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

FILE_PATH = r"C:\Users\Admin\Downloads\06_2521001108_PRJ04_Phan Thị Quế Trân (1).csv"
JUNK_VALS = ['unknown', '?', '-', 'none', 'nan', 'null', '']

print("=" * 85)
print("CHƯƠNG TRÌNH PYTHON PHÂN TÍCH TOÀN DIỆN TẬP DỮ LIỆU NGOẠI TRÚ (OUTPATIENT)")
print(f"File nguồn: {FILE_PATH}")
print("=" * 85)

# Đọc dữ liệu
print("\nĐang nạp dữ liệu vào bộ nhớ RAM...")
df = pd.read_csv(FILE_PATH, low_memory=False)
total_rows, total_cols = df.shape
memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"=> Nạp thành công: {total_rows:,} dòng, {total_cols} cột (Dung lượng RAM: {memory_mb:.2f} MB)")

# ------------------------------------------------------------------------------
# PHẦN 1: CODE PHÂN TÍCH NGUỒN GỐC DỮ LIỆU (DATA PROVENANCE & CONTEXT)
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("PHẦN 1: PHÂN TÍCH NGUỒN GỐC DỮ LIỆU (DATA PROVENANCE)")
print("=" * 85)

# 1.1. Kiểm tra các mẫu tiền tố định danh đặc trưng của hệ thống CMS / DE-SynPUF
id_checks = {
    'BeneID (Người thụ hưởng)': df['BeneID'].dropna().astype(str).str.upper().str.startswith('BENE').mean(),
    'ClaimID (Hồ sơ bồi thường)': df['ClaimID'].dropna().astype(str).str.upper().str.startswith('CLM').mean(),
    'Provider (Cơ sở y tế)': df['Provider'].dropna().astype(str).str.upper().str.startswith('PRV').mean(),
    'AttendingPhysician (Bác sĩ)': df['AttendingPhysician'].dropna().astype(str).str.upper().str.startswith('PHY').mean(),
}

print("1.1. Kiểm chứng cấu trúc tiền tố chuẩn hóa theo chuẩn CMS Medicare:")
for k, v in id_checks.items():
    print(f"  + {k:<30}: {v:.2%} tuân theo định dạng chuẩn (BENE/CLM/PRV/PHY)")

# 1.2. Kiểm tra khoảng thời gian phát sinh hồ sơ
s_dt = pd.to_datetime(df['ClaimStartDt'], errors='coerce').dropna()
e_dt = pd.to_datetime(df['ClaimEndDt'], errors='coerce').dropna()
print(f"\n1.2. Khoảng thời gian phát sinh giao dịch (Coverage Window):")
print(f"  + Ngày bắt đầu sớm nhất (Min ClaimStartDt) : {s_dt.min().strftime('%Y-%m-%d')}")
print(f"  + Ngày bắt đầu muộn nhất (Max ClaimStartDt): {s_dt.max().strftime('%Y-%m-%d')}")
print(f"  + Ngày kết thúc sớm nhất (Min ClaimEndDt)   : {e_dt.min().strftime('%Y-%m-%d')}")
print(f"  + Ngày kết thúc muộn nhất (Max ClaimEndDt)  : {e_dt.max().strftime('%Y-%m-%d')}")
print("  => Căn cứ thực nghiệm: Thời gian nằm trọn trong giai đoạn 2008 - 2009 của bộ CMS DE-SynPUF.")

# 1.3. Kiểm tra tính bảo mật Non-PHI (Protected Health Information)
phi_keywords = ['name', 'ssn', 'social', 'address', 'phone', 'email', 'zip', 'age', 'gender', 'dob', 'birth']
found_phi = [c for c in df.columns if any(k in c.lower() for k in phi_keywords)]
print(f"\n1.3. Kiểm tra bảo mật dữ liệu cá nhân (HIPAA / Non-PHI):")
print(f"  + Các biến định danh nhạy cảm tìm thấy: {found_phi if found_phi else 'KHÔNG CÓ (0 biến)'}")
print("  => Kết luận: Dữ liệu mô phỏng tổng hợp (Synthetic Data), hoàn toàn an toàn, phi định danh cá nhân.")


CHƯƠNG TRÌNH PYTHON PHÂN TÍCH TOÀN DIỆN TẬP DỮ LIỆU NGOẠI TRÚ (OUTPATIENT)
File nguồn: C:\Users\Admin\Downloads\06_2521001108_PRJ04_Phan Thị Quế Trân (1).csv

Đang nạp dữ liệu vào bộ nhớ RAM...
=> Nạp thành công: 517,737 dòng, 27 cột (Dung lượng RAM: 567.69 MB)

PHẦN 1: PHÂN TÍCH NGUỒN GỐC DỮ LIỆU (DATA PROVENANCE)
1.1. Kiểm chứng cấu trúc tiền tố chuẩn hóa theo chuẩn CMS Medicare:
  + BeneID (Người thụ hưởng)      : 79.45% tuân theo định dạng chuẩn (BENE/CLM/PRV/PHY)
  + ClaimID (Hồ sơ bồi thường)    : 75.98% tuân theo định dạng chuẩn (BENE/CLM/PRV/PHY)
  + Provider (Cơ sở y tế)         : 76.12% tuân theo định dạng chuẩn (BENE/CLM/PRV/PHY)
  + AttendingPhysician (Bác sĩ)   : 92.77% tuân theo định dạng chuẩn (BENE/CLM/PRV/PHY)

1.2. Khoảng thời gian phát sinh giao dịch (Coverage Window):
  + Ngày bắt đầu sớm nhất (Min ClaimStartDt) : 2008-12-12
  + Ngày bắt đầu muộn nhất (Max ClaimStartDt): 2009-12-31
  + Ngày kết thúc sớm nhất (Min ClaimEndDt)   : 2009-01-01
  + Ngày kết thúc muộn nhấ

In [2]:
# ------------------------------------------------------------------------------
# PHẦN 2: CODE PHÂN TÍCH QUY TRÌNH PHÁT SINH DỮ LIỆU (DATA GENERATION LIFECYCLE)
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("PHẦN 2: QUY TRÌNH PHÁT SINH DỮ LIỆU (TỪ KHÁM BỆNH ĐẾN HỒ SƠ BỒI THƯỜNG)")
print("=" * 85)

# 2.1. Phân tích thời lượng khám bệnh (Đặc trưng Ngoại trú - Không qua đêm)
durations = (e_dt - s_dt).dt.days + 1
durations = durations[durations >= 1]
print("2.1. Bước 1: Tiếp nhận & Thời lượng thăm khám (ClaimDuration):")
print(f"  + Tỷ lệ khám và giải quyết trong ngày (1 ngày) : {(durations == 1).mean():.2%}")
print(f"  + Thời lượng trung bình                       : {durations.mean():.2f} ngày")
print(f"  + Thời lượng trung vị (Median)                : {durations.median():.0f} ngày (Max: {durations.max():.0f} ngày)")
print("  => Minh chứng: Đặc thù ngoại trú đa số (87.54%) kết thúc điều trị trong ngày.")

# 2.2. Phân tích nhân sự y tế tham gia điều trị
print("\n2.2. Bước 2: Phân công nhân sự y tế điều trị & can thiệp:")
for col, role in [
    ('AttendingPhysician', 'Bác sĩ điều trị chính (Chịu trách nhiệm ca bệnh)'),
    ('OperatingPhysician', 'Bác sĩ phẫu thuật / thực hiện thủ thuật'),
    ('OtherPhysician',     'Bác sĩ hỗ trợ / phụ mổ / hội chẩn')
]:
    clean_cnt = (~df[col].astype(str).str.strip().str.lower().isin(JUNK_VALS)).sum()
    print(f"  + {role:<45}: {clean_cnt:,} ca ({clean_cnt/len(df):.2%})")

# 2.3. Phân tích mã hóa lâm sàng (ICD-9 Chẩn đoán và ICD-9-CM Thủ thuật)
diag_cols = [c for c in df.columns if c.startswith('ClmDiagnosisCode_')]
proc_cols = [c for c in df.columns if c.startswith('ClmProcedureCode_')]

diag_clean_mask = df[diag_cols].astype(str).apply(lambda c: ~c.str.strip().str.lower().isin(JUNK_VALS))
proc_clean_mask = df[proc_cols].astype(str).apply(lambda c: ~c.str.strip().str.lower().isin(JUNK_VALS))

print("\n2.3. Bước 3: Mã hóa bệnh án điện tử (Medical Coding):")
print(f"  + Số mã chẩn đoán ICD-9 trung bình mỗi hồ sơ  : {diag_clean_mask.sum(axis=1).mean():.2f} / 10 mã")
print(f"  + Số mã thủ thuật ICD-9-CM trung bình mỗi ca  : {proc_clean_mask.sum(axis=1).mean():.2f} / 6 mã")

# 2.4. Phân tích cơ chế tài chính (Thẩm định & Quyết toán bồi hoàn)
def extract_num(s):
    return pd.to_numeric(s.astype(str).str.replace(',', '').str.extract(r'([-+]?\d*\.?\d+)')[0], errors='coerce')

reimbursed = extract_num(df['InscClaimAmtReimbursed'])
deductible = extract_num(df['DeductibleAmtPaid'])

print("\n2.4. Bước 4: Thẩm định bồi hoàn và Quyết toán tài chính:")
print(f"  + Số tiền bảo hiểm bồi hoàn (InscClaimAmtReimbursed) : Trung vị = {reimbursed.median():,.2f} USD | Trung bình = {reimbursed.mean():,.2f} USD")
print(f"  + Tiền khấu trừ người bệnh tự trả (DeductibleAmtPaid): Trung vị = {deductible.median():,.2f} USD | Trung bình = {deductible.mean():,.2f} USD")
print(f"  + Tỷ lệ hồ sơ có bồi hoàn = 0 USD                    : {(reimbursed == 0).mean():.2%}")



PHẦN 2: QUY TRÌNH PHÁT SINH DỮ LIỆU (TỪ KHÁM BỆNH ĐẾN HỒ SƠ BỒI THƯỜNG)
2.1. Bước 1: Tiếp nhận & Thời lượng thăm khám (ClaimDuration):
  + Tỷ lệ khám và giải quyết trong ngày (1 ngày) : 87.54%
  + Thời lượng trung bình                       : 2.42 ngày
  + Thời lượng trung vị (Median)                : 1 ngày (Max: 24 ngày)
  => Minh chứng: Đặc thù ngoại trú đa số (87.54%) kết thúc điều trị trong ngày.

2.2. Bước 2: Phân công nhân sự y tế điều trị & can thiệp:
  + Bác sĩ điều trị chính (Chịu trách nhiệm ca bệnh): 508,254 ca (98.17%)
  + Bác sĩ phẫu thuật / thực hiện thủ thuật      : 426,666 ca (82.41%)
  + Bác sĩ hỗ trợ / phụ mổ / hội chẩn            : 426,291 ca (82.34%)

2.3. Bước 3: Mã hóa bệnh án điện tử (Medical Coding):
  + Số mã chẩn đoán ICD-9 trung bình mỗi hồ sơ  : 9.41 / 10 mã
  + Số mã thủ thuật ICD-9-CM trung bình mỗi ca  : 5.52 / 6 mã

2.4. Bước 4: Thẩm định bồi hoàn và Quyết toán tài chính:
  + Số tiền bảo hiểm bồi hoàn (InscClaimAmtReimbursed) : Trung vị = 80.00 USD | T

In [3]:
# ------------------------------------------------------------------------------
# PHẦN 3: CODE PHÂN TÍCH ĐƠN VỊ QUAN SÁT ("MỖI DÒNG LÀ GÌ?")
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("PHẦN 3: PHÂN TÍCH ĐƠN VỊ QUAN SÁT (UNIT OF OBSERVATION: 'MỖI DÒNG LÀ GÌ?')")
print("=" * 85)

# 3.1. Phân tích tính duy nhất và bản chất của mã ClaimID
valid_claims = df['ClaimID'].dropna().astype(str).str.strip()
real_claims = valid_claims[valid_claims.str.upper().str.startswith('CLM')]
duplicate_real_claims = real_claims.duplicated().sum()

print("3.1. Bản chất đơn vị hồ sơ bồi thường (Claim-level Unit):")
print(f"  + Tổng số mã ClaimID có định dạng chuẩn (CLM...): {len(real_claims):,}")
print(f"  + Số lượng mã ClaimID duy nhất                  : {real_claims.nunique():,}")
print(f"  + Tỷ lệ khuyết thiếu/nhiễu ở cột ClaimID       : {((len(df) - len(real_claims))/len(df)):.2%}")
print("  => PHÂN TÍCH BẢN CHẤT MỖI DÒNG:")
print("     - Mỗi dòng đại diện cho MỘT ĐỢT YÊU CẦU BỒI THƯỜNG BẢO HIỂM Y TẾ (Single Medical Claim Encounter).")
print("     - Mỗi dòng có thông tin bệnh nhân (BeneID), cơ sở y tế (Provider), thời gian (StartDt/EndDt),")
print("       bác sĩ phụ trách, chẩn đoán ICD-9 và số tiền bồi hoàn độc lập.")
print("     - Cột ClaimID bị nhiễu tổng hợp (synthetic noise do giảng viên cài cắm để kiểm tra kỹ năng tiền xử lý),")
print("       khiến một số mã CLM bị lặp giữa các bệnh nhân khác nhau, nhưng toàn bộ các dòng là các giao dịch độc lập.")

# 3.2. Chứng minh dữ liệu KHÔNG PHẢI là cấp độ Bệnh nhân (1-N Relationship)
valid_bene = df['BeneID'].dropna().astype(str).str.strip()
real_bene = valid_bene[valid_bene.str.upper().str.startswith('BENE')]
claims_per_patient = real_bene.value_counts()

print("\n3.2. Mối quan hệ 1 - Nhiều (1 - N) giữa Người bệnh (BeneID) và Hồ sơ (ClaimID):")
print(f"  + Tổng số người thụ hưởng duy nhất (BeneID)    : {len(claims_per_patient):,} người")
print(f"  + Số hồ sơ bồi thường trung bình mỗi bệnh nhân : {claims_per_patient.mean():.2f} hồ sơ")
print(f"  + Tối thiểu / Tối đa hồ sơ của 1 bệnh nhân     : {claims_per_patient.min()} hồ sơ / {claims_per_patient.max()} hồ sơ")
print(f"  + Tỷ lệ bệnh nhân có nhiều hơn 1 đợt khám      : {(claims_per_patient > 1).mean():.2%}")
print("  => KHẲNG ĐỊNH: Dữ liệu KHÔNG PHẢI ở cấp độ bệnh nhân (Patient-level).")

# 3.3. Mối quan hệ với Cơ sở y tế (Provider)
valid_prv = df['Provider'].dropna().astype(str).str.strip()
real_prv = valid_prv[valid_prv.str.upper().str.startswith('PRV')]
claims_per_prv = real_prv.value_counts()
print(f"\n3.3. Mối quan hệ với Cơ sở y tế (Provider):")
print(f"  + Tổng số cơ sở y tế (Provider)                : {len(claims_per_prv):,} cơ sở")
print(f"  + Số hồ sơ trung bình mỗi cơ sở phục vụ        : {claims_per_prv.mean():.2f} hồ sơ (Max = {claims_per_prv.max():,} hồ sơ)")



PHẦN 3: PHÂN TÍCH ĐƠN VỊ QUAN SÁT (UNIT OF OBSERVATION: 'MỖI DÒNG LÀ GÌ?')
3.1. Bản chất đơn vị hồ sơ bồi thường (Claim-level Unit):
  + Tổng số mã ClaimID có định dạng chuẩn (CLM...): 383,618
  + Số lượng mã ClaimID duy nhất                  : 336,017
  + Tỷ lệ khuyết thiếu/nhiễu ở cột ClaimID       : 25.90%
  => PHÂN TÍCH BẢN CHẤT MỖI DÒNG:
     - Mỗi dòng đại diện cho MỘT ĐỢT YÊU CẦU BỒI THƯỜNG BẢO HIỂM Y TẾ (Single Medical Claim Encounter).
     - Mỗi dòng có thông tin bệnh nhân (BeneID), cơ sở y tế (Provider), thời gian (StartDt/EndDt),
       bác sĩ phụ trách, chẩn đoán ICD-9 và số tiền bồi hoàn độc lập.
     - Cột ClaimID bị nhiễu tổng hợp (synthetic noise do giảng viên cài cắm để kiểm tra kỹ năng tiền xử lý),
       khiến một số mã CLM bị lặp giữa các bệnh nhân khác nhau, nhưng toàn bộ các dòng là các giao dịch độc lập.

3.2. Mối quan hệ 1 - Nhiều (1 - N) giữa Người bệnh (BeneID) và Hồ sơ (ClaimID):
  + Tổng số người thụ hưởng duy nhất (BeneID)    : 140,646 người
  + Số hồ sơ 

In [4]:
# ------------------------------------------------------------------------------
# PHẦN 4: DANH SÁCH 27 BIẾN CỦA FILE NGOẠI TRÚ (FULL DATA DICTIONARY)
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("PHẦN 4: BẢNG TỪ ĐIỂN DỮ LIỆU VÀ ĐẶC TÍNH 27 BIẾN CỦA TỆP NGOẠI TRÚ")
print("=" * 85)

def categorize_column(col_name):
    if col_name in ['BeneID', 'ClaimID', 'Provider']:
        return "1. Định danh (ID/Key)"
    elif 'Dt' in col_name:
        return "2. Thời gian (Temporal)"
    elif 'Amt' in col_name or 'Paid' in col_name:
        return "3. Tài chính (Financial)"
    elif 'Physician' in col_name:
        return "4. Nhân sự y tế (Staffing)"
    elif 'Diagnosis' in col_name or 'Procedure' in col_name:
        return "5. Lâm sàng (ICD-9/Proc)"
    else:
        return "6. Khác"

dict_records = []
for idx, col in enumerate(df.columns, 1):
    raw_s = df[col]
    clean_s = raw_s.astype(str).str.strip()
    is_junk = clean_s.str.lower().isin(JUNK_VALS) | raw_s.isna()
    
    missing_count = is_junk.sum()
    missing_pct = (missing_count / total_rows) * 100
    unique_count = raw_s[~is_junk].nunique()
    
    sample_vals = raw_s[~is_junk].head(3).tolist()
    sample_str = ", ".join([str(v) for v in sample_vals])
    
    dict_records.append({
        'STT': idx,
        'Tên Biến': col,
        'Nhóm Nghiệp Vụ': categorize_column(col),
        'Kiểu Dữ Liệu': str(raw_s.dtype),
        'Số Unique': f"{unique_count:,}",
        'Tỷ Lệ Thiếu (%)': f"{missing_pct:.2f}%",
        'Giá Trị Mẫu': sample_str[:30]
    })

dict_df = pd.DataFrame(dict_records)

# In bảng từ điển chi tiết
print(dict_df.to_string(index=False))

# Xuất bảng từ điển ra file CSV
output_dict_path = os.path.join(r"C:\Users\Admin\Downloads\Aa", "tu_dien_27_bien_ngoai_tru.csv")
dict_df.to_csv(output_dict_path, index=False, encoding='utf-8-sig')
print(f"\n[XUẤT FILE THÀNH CÔNG] Đã lưu từ điển 27 biến tại: {output_dict_path}")
print("=" * 85)



PHẦN 4: BẢNG TỪ ĐIỂN DỮ LIỆU VÀ ĐẶC TÍNH 27 BIẾN CỦA TỆP NGOẠI TRÚ
 STT               Tên Biến             Nhóm Nghiệp Vụ Kiểu Dữ Liệu Số Unique Tỷ Lệ Thiếu (%)                    Giá Trị Mẫu
   1                 BeneID      1. Định danh (ID/Key)          str   141,048          26.30% BENE11002, bene11003,   BENE11
   2                ClaimID      1. Định danh (ID/Key)          str   336,018          22.65% CLM624349, CLM189947, CLM43802
   3           ClaimStartDt    2. Thời gian (Temporal)          str       399           5.83% 2009-10-11, 2009-02-12, 2009-0
   4             ClaimEndDt    2. Thời gian (Temporal)          str       378           7.45% 2009-10-11, 2009-02-12, 2009-0
   5               Provider      1. Định danh (ID/Key)          str     8,832          25.98%   PRV56011, PRV57610, PRV57595
   6 InscClaimAmtReimbursed   3. Tài chính (Financial)          str       890           4.75%                     30, 80, 10
   7     AttendingPhysician 4. Nhân sự y tế (Staffing)   

In [5]:
# ==============================================================================
# QUY TRÌNH TIỀN XỬ LÝ & TỐI ƯU BỘ NHỚ RAM (DÀNH RIÊNG CHO NGOẠI TRÚ)
# ==============================================================================
import re
import pandas as pd
import numpy as np

# 1. ĐƯỜNG DẪN DỮ LIỆU
f_out = r"C:\Users\Admin\Downloads\06_2521001108_PRJ04_Phan Thị Quế Trân (1).csv"

# 2. ĐỌC DỮ LIỆU VÀ CHUẨN HÓA TÊN CỘT
print("[1/3] Đang nạp tệp dữ liệu Ngoại trú...")
df = pd.read_csv(f_out, low_memory=False)

# Chuẩn hóa tên cột (loại bỏ khoảng trắng thừa)
df.columns = df.columns.str.strip()
print(f"  + Kích thước dữ liệu Ngoại trú: {df.shape[0]:,} dòng × {df.shape[1]} cột")

# Ghi nhận dung lượng RAM ban đầu
mem_raw_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"  + Dung lượng RAM ban đầu     : {mem_raw_mb:.2f} MB")

# 3. THÊM NHÃN LOẠI HÌNH (TÙY CHỌN ĐỂ ĐỒNG BỘ DỮ LIỆU)
df['ClaimType'] = 'Outpatient'

# 4. TỐI ƯU HÓA KIỂU DỮ LIỆU ĐỂ GIẢM BỘ NHỚ RAM
print("\n[2/3] Đang tối ưu hóa kiểu dữ liệu (Downcasting & Categoricals)...")

# (A) Làm sạch tiền tệ và ép sang float32 (tiết kiệm 50% RAM)
def clean_currency_to_float32(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.replace(',', '', regex=False).str.strip()
    extracted = s.str.extract(r'([-+]?\d*\.?\d+)')[0]
    return pd.to_numeric(extracted, errors='coerce').astype('float32')

df['InscClaimAmtReimbursed'] = clean_currency_to_float32(df['InscClaimAmtReimbursed'])
df['DeductibleAmtPaid']      = clean_currency_to_float32(df['DeductibleAmtPaid'])

# (B) Chuyển đổi các cột ngày tháng (chỉ lấy 2 cột thực sự có ở Ngoại trú)
date_cols = ['ClaimStartDt', 'ClaimEndDt']
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# (C) Chuyển các cột danh mục lặp lại sang kiểu 'category'
cat_cols = [
    'ClaimType', 'Provider',
    'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician',
    'ClmAdmitDiagnosisCode'
]
cat_cols += [f'ClmProcedureCode_{i}' for i in range(1, 7)]
cat_cols += [f'ClmDiagnosisCode_{i}' for i in range(1, 11)]

for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].astype('category')

# (D) Giữ nguyên BeneID, ClaimID dạng string
df['BeneID']  = df['BeneID'].astype(str)
df['ClaimID'] = df['ClaimID'].astype(str)

# 5. ĐÁNH GIÁ HIỆU QUẢ TIẾT KIỆM RAM
mem_opt_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
saved_pct  = (1 - mem_opt_mb / mem_raw_mb) * 100

print("\n" + "=" * 65)
print(f"KẾT QUẢ TỐI ƯU HÓA: {mem_raw_mb:.2f} MB  -->  {mem_opt_mb:.2f} MB")
print(f"TIẾT KIỆM ĐƯỢC    : {mem_raw_mb - mem_opt_mb:.2f} MB ({saved_pct:.2f}%)")
print("=" * 65)

[1/3] Đang nạp tệp dữ liệu Ngoại trú...
  + Kích thước dữ liệu Ngoại trú: 517,737 dòng × 27 cột
  + Dung lượng RAM ban đầu     : 567.69 MB

[2/3] Đang tối ưu hóa kiểu dữ liệu (Downcasting & Categoricals)...

KẾT QUẢ TỐI ƯU HÓA: 567.69 MB  -->  97.93 MB
TIẾT KIỆM ĐƯỢC    : 469.76 MB (82.75%)


In [6]:
# ==============================================================================
# ĐỀ TÀI PRJ-04: KIỂM TOÁN VÀ LÀM SẠCH DỮ LIỆU BẨN TOÀN DIỆN (DATA CLEANING)
# Tệp thực hiện: C:\Users\Admin\Downloads\06_2521001108_PRJ04_Phan Thị Quế Trân (1).csv
# Nhóm 6: Nguyễn Ngọc Phương Thảo (2521001099) & Phan Thị Quế Trân (2521001108)
# ==============================================================================

import sys
import os
import re
import pandas as pd
import numpy as np

# Cấu hình UTF-8 chống lỗi font console trên Windows
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

INPUT_PATH = r"C:\Users\Admin\Downloads\06_2521001108_PRJ04_Phan Thị Quế Trân (1).csv"
OUTPUT_DIR = r"C:\Users\Admin\Downloads\Aa"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 85)
print("HỆ THỐNG PHÁT HIỆN, KIỂM TOÁN VÀ LÀM SẠCH DỮ LIỆU BẨN (DATA QUALITY AUDIT)")
print("=" * 85)

# Danh sách toàn bộ các biến thể dữ liệu thiếu giả lập
JUNK_MISSING_STRINGS = [
    'none', 'n/a', 'na', '?', '-', 'null', 'nan', 'unknown', 'chưa rõ',
    'undefined', 'missing', ''
]

# ------------------------------------------------------------------------------
# 1. ĐỌC DỮ LIỆU THÔ VÀ KIỂM TOÁN DỮ LIỆU BẨN
# ------------------------------------------------------------------------------
print("\n[BƯỚC 1] Đọc dữ liệu thô và tiến hành kiểm toán dữ liệu bẩn...")
# keep_default_na=False để giữ nguyên vẹn các chuỗi rác như '?', '-', 'unknown'
df_raw = pd.read_csv(INPUT_PATH, low_memory=False, keep_default_na=False)
total_rows = len(df_raw)
print(f"=> Tổng số bản ghi ban đầu: {total_rows:,} dòng, {df_raw.shape[1]} cột")

# 1.1. Kiểm tra trùng lặp hoàn toàn (Full duplicate rows)
full_dups = df_raw.duplicated().sum()
print(f"\n1.1. Kiểm tra trùng lặp bản ghi (Duplicates):")
print(f"  + Trùng lặp hoàn toàn trên toàn bộ 27 cột : {full_dups} dòng (0.00%)")
cid_clean_raw = df_raw['ClaimID'].str.strip()
print(f"  + Số mã ClaimID bị lặp lại ở dữ liệu thô   : {cid_clean_raw.duplicated().sum():,} lần")
print("    (Nguyên nhân: Do các chuỗi khuyết giả lập như 'unknown', '?', '-' bị lặp lại hàng nghìn lần)")

# 1.2. Kiểm tra khoảng trắng thừa và không nhất quán chữ hoa/thường
print(f"\n1.2. Kiểm tra nhãn không nhất quán và khoảng trắng thừa:")
ws_issues = {}
for c in df_raw.columns:
    count_ws = (df_raw[c].astype(str) != df_raw[c].astype(str).str.strip()).sum()
    if count_ws > 0:
        ws_issues[c] = count_ws
print(f"  + Số lượng cột bị dính khoảng trắng đầu/cuối: {len(ws_issues)} / 27 cột")
print(f"  + Điển hình: ClaimID ({ws_issues.get('ClaimID', 0):,} dòng), BeneID ({ws_issues.get('BeneID', 0):,} dòng)")

# Thống kê chữ thường lẫn lộn
bene_lower = df_raw['BeneID'].str.contains(r'[a-z]').sum()
clm_lower  = df_raw['ClaimID'].str.contains(r'[a-z]').sum()
prv_lower  = df_raw['Provider'].str.contains(r'[a-z]').sum()
print(f"  + Mã BeneID dính chữ thường  (bene...)   : {bene_lower:,} dòng ({bene_lower/total_rows:.2%})")
print(f"  + Mã ClaimID dính chữ thường (clm...)    : {clm_lower:,} dòng ({clm_lower/total_rows:.2%})")
print(f"  + Mã Provider dính chữ thường (prv...)   : {prv_lower:,} dòng ({prv_lower/total_rows:.2%})")

# 1.3. Kiểm toán dị thường số học ở các cột tiền tệ
print(f"\n1.3. Kiểm toán dị thường số học (Giá trị số âm trong cột tiền):")
def parse_financial_series(series):
    s = series.astype(str).str.replace(',', '', regex=False).str.strip()
    extracted = s.str.extract(r'([-+]?\d*\.?\d+)')[0]
    return pd.to_numeric(extracted, errors='coerce'), s.str.contains(r'[A-Za-z]', regex=True).sum()

reimbursed_num, reimbursed_dirty_units = parse_financial_series(df_raw['InscClaimAmtReimbursed'])
deductible_num, deductible_dirty_units = parse_financial_series(df_raw['DeductibleAmtPaid'])

neg_reimbursed = (reimbursed_num < 0).sum()
neg_deductible = (deductible_num < 0).sum()

print(f"  + Tiền bồi thường bảo hiểm (InscClaimAmtReimbursed):")
print(f"    - Bị dính đơn vị đo lường/tiền tệ (USD, kg)    : {reimbursed_dirty_units:,} dòng")
print(f"    - Mang giá trị số ÂM (< 0)                     : {neg_reimbursed:,} dòng ({neg_reimbursed/total_rows:.2%})")
print(f"    - Giá trị âm thấp nhất                         : {reimbursed_num[reimbursed_num < 0].min():,.2f} USD")
print(f"  + Tiền khấu trừ người bệnh (DeductibleAmtPaid):")
print(f"    - Bị dính đơn vị đo lường/tiền tệ (USD, kg)    : {deductible_dirty_units:,} dòng")
print(f"    - Mang giá trị số ÂM (< 0)                     : {neg_deductible:,} dòng ({neg_deductible/total_rows:.2%})")
print(f"    - Giá trị âm thấp nhất                         : {deductible_num[deductible_num < 0].min():,.2f} USD")

# 1.4. Phát hiện giá trị ngoại lai (Outliers) theo tiêu chuẩn IQR
valid_pos_reimbursed = reimbursed_num[reimbursed_num >= 0].dropna()
q1 = valid_pos_reimbursed.quantile(0.25)
q2 = valid_pos_reimbursed.quantile(0.50)
q3 = valid_pos_reimbursed.quantile(0.75)
iqr = q3 - q1
upper_outlier_bound = q3 + 1.5 * iqr
outliers_count = (valid_pos_reimbursed > upper_outlier_bound).sum()

print(f"\n1.4. Kiểm toán giá trị ngoại lai (IQR Outlier Detection):")
print(f"  + Q1 = {q1:.2f} USD | Trung vị (Q2) = {q2:.2f} USD | Q3 = {q3:.2f} USD")
print(f"  + Khoảng tứ phân vị (IQR)                       : {iqr:.2f} USD")
print(f"  + Ngưỡng ngoại lai trên (Q3 + 1.5*IQR)           : {upper_outlier_bound:.2f} USD")
print(f"  + Số lượng hồ sơ vượt ngưỡng ngoại lai          : {outliers_count:,} hồ sơ ({outliers_count/len(valid_pos_reimbursed):.2%})")
print(f"  + Giá trị bồi hoàn lớn nhất (Max)               : {valid_pos_reimbursed.max():,.2f} USD")

# ------------------------------------------------------------------------------
# 2. THỰC THI QUY TRÌNH LÀM SẠCH DỮ LIỆU (DATA CLEANING PIPELINE)
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("[BƯỚC 2] THỰC THI TIỀN XỬ LÝ VÀ LÀM SẠCH DỮ LIỆU CHUẨN MỰC")
print("=" * 85)

df_clean = df_raw.copy()

# Thao tác 1: Cắt tỉa toàn bộ khoảng trắng thừa trên 27 cột
print("  (1) Cắt tỉa khoảng trắng thừa (Leading/Trailing Whitespace)...")
for c in df_clean.columns:
    df_clean[c] = df_clean[c].astype(str).str.strip()

# Thao tác 2: Đồng nhất chữ hoa cho các cột mã định danh
print("  (2) Đồng nhất viết hoa (Uppercase) cho BeneID, ClaimID, Provider, Bác sĩ...")
id_columns = ['BeneID', 'ClaimID', 'Provider', 'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician']
for c in id_columns:
    df_clean[c] = df_clean[c].str.upper()

# Thao tác 3: Quy đổi dứt điểm các chuỗi khuyết giả lập về np.nan
print("  (3) Chuẩn hóa các chuỗi khuyết giả lập ('?', '-', 'unknown', 'none', '') về np.nan...")
for c in df_clean.columns:
    mask_junk = df_clean[c].str.lower().isin(JUNK_MISSING_STRINGS)
    df_clean.loc[mask_junk, c] = np.nan

# Thao tác 4: Làm sạch chuỗi tiền tệ bằng Regex và ép kiểu float32
print("  (4) Bóc tách số thuần từ chuỗi lẫn đơn vị tiền tệ bằng Regex...")
df_clean['InscClaimAmtReimbursed'] = pd.to_numeric(
    df_clean['InscClaimAmtReimbursed'].astype(str).str.replace(',', '').str.extract(r'([-+]?\d*\.?\d+)')[0],
    errors='coerce'
)
df_clean['DeductibleAmtPaid'] = pd.to_numeric(
    df_clean['DeductibleAmtPaid'].astype(str).str.replace(',', '').str.extract(r'([-+]?\d*\.?\d+)')[0],
    errors='coerce'
)

# Thao tác 5: Chuẩn hóa ngày tháng (xử lý ngày cắt cụt '2009-08-' về NaT)
print("  (5) Chuẩn hóa ngày tháng (ClaimStartDt, ClaimEndDt)...")
df_clean['ClaimStartDt'] = pd.to_datetime(df_clean['ClaimStartDt'], errors='coerce')
df_clean['ClaimEndDt']   = pd.to_datetime(df_clean['ClaimEndDt'], errors='coerce')

# Thao tác 6: Tách và cô lập dị thường số âm
print("  (6) Cô lập các bản ghi mang giá trị số âm sang tập riêng...")
# Tách các dòng có tiền bồi hoàn mang số âm
mask_negative = df_clean['InscClaimAmtReimbursed'] < 0
df_negative_anomalies = df_clean[mask_negative].copy()

# Giữ lại tập dữ liệu hợp lệ phục vụ định phí bồi thường (>= 0)
df_clean_valid = df_clean[df_clean['InscClaimAmtReimbursed'] >= 0].copy()

# Tiền khấu trừ nếu âm cũng được clip về 0
df_clean_valid['DeductibleAmtPaid'] = df_clean_valid['DeductibleAmtPaid'].clip(lower=0)

print(f"\n=> TỔNG KẾT KẾT QUẢ LÀM SẠCH:")
print(f"  + Tổng số hồ sơ ban đầu                        : {total_rows:,} dòng")
print(f"  + Số hồ sơ có chi phí ÂM cô lập kiểm toán      : {len(df_negative_anomalies):,} dòng (1.42%)")
print(f"  + Số hồ sơ hợp lệ thu hồi phục vụ mô hình hóa  : {len(df_clean_valid):,} dòng ({len(df_clean_valid)/total_rows:.2%})")

# ------------------------------------------------------------------------------
# 3. XUẤT CÁC TỆP DỮ LIỆU ĐÃ LÀM SẠCH VÀ BẢNG BÁO CÁO
# ------------------------------------------------------------------------------
print("\n[BƯỚC 3] Xuất các tệp kết quả ra thư mục làm việc...")

# Lưu tập hợp lệ phục vụ huấn luyện mô hình
clean_valid_path = os.path.join(OUTPUT_DIR, "df_outpatient_clean.csv")
df_clean_valid.to_csv(clean_valid_path, index=False, encoding='utf-8-sig')
print(f"  + Đã lưu tệp sạch hợp lệ tại         : {clean_valid_path}")

# Lưu tập dị thường số âm để phục vụ phân tích kiểm toán
neg_path = os.path.join(OUTPUT_DIR, "df_negative_claims.csv")
df_negative_anomalies.to_csv(neg_path, index=False, encoding='utf-8-sig')
print(f"  + Đã lưu tệp dị thường số âm tại     : {neg_path}")

# Bảng tóm tắt kết quả kiểm toán làm sạch
audit_summary = pd.DataFrame({
    'Danh Mục Kiểm Toán': [
        'Trùng lặp hoàn toàn (Full Duplicates)',
        'Cột dính khoảng trắng thừa',
        'Số mã BeneID bị viết thường',
        'Số tiền dính đơn vị đo lường (USD, kg)',
        'Số bản ghi mang giá trị số âm (< 0)',
        'Ngưỡng ngoại lai trên (IQR Upper Bound)',
        'Số lượng hồ sơ ngoại lai (> 690 USD)',
        'Số hồ sơ hợp lệ sau cùng (Trainable)'
    ],
    'Số Lượng Phát Hiện': [
        '0 dòng (0%)',
        '25 / 27 cột bị dính khoảng trắng',
        '106,143 dòng (20.50%)',
        '32,361 dòng bồi thường | 40,312 dòng khấu trừ',
        '7,344 dòng (1.42%)',
        '690.00 USD (Q3 + 1.5*IQR)',
        '62,457 dòng (12.86%)',
        '485,776 dòng (93.83%)'
    ],
    'Phương Pháp Xử Lý Đã Áp Dụng': [
        'Không phát sinh dòng trùng 100%',
        'Xử lý bằng .str.strip() trên toàn bộ 27 cột',
        'Đồng nhất bằng .str.upper()',
        'Bóc tách số thực thuần bằng Regex ([-+]?\\d*\\.?\\d+)',
        'Cô lập xuất riêng file df_negative_claims.csv',
        'Giữ nguyên ngoại lai vì bản chất đuôi dày bảo hiểm',
        'Dùng mô hình GLM Tweedie và Log-transform xử lý',
        'Sẵn sàng đưa vào Feature Engineering và Model'
    ]
})

summary_report_path = os.path.join(OUTPUT_DIR, "bao_cao_kiem_toan_lam_sach.csv")
audit_summary.to_csv(summary_report_path, index=False, encoding='utf-8-sig')
print(f"  + Đã lưu bảng báo cáo kiểm toán tại : {summary_report_path}")
print("\n" + "=" * 85)
print("HOÀN TẤT TOÀN DIỆN QUY TRÌNH KIỂM TOÁN VÀ LÀM SẠCH DỮ LIỆU!")
print("=" * 85)


HỆ THỐNG PHÁT HIỆN, KIỂM TOÁN VÀ LÀM SẠCH DỮ LIỆU BẨN (DATA QUALITY AUDIT)

[BƯỚC 1] Đọc dữ liệu thô và tiến hành kiểm toán dữ liệu bẩn...
=> Tổng số bản ghi ban đầu: 517,737 dòng, 27 cột

1.1. Kiểm tra trùng lặp bản ghi (Duplicates):
  + Trùng lặp hoàn toàn trên toàn bộ 27 cột : 0 dòng (0.00%)
  + Số mã ClaimID bị lặp lại ở dữ liệu thô   : 181,712 lần
    (Nguyên nhân: Do các chuỗi khuyết giả lập như 'unknown', '?', '-' bị lặp lại hàng nghìn lần)

1.2. Kiểm tra nhãn không nhất quán và khoảng trắng thừa:
  + Số lượng cột bị dính khoảng trắng đầu/cuối: 25 / 27 cột
  + Điển hình: ClaimID (54,039 dòng), BeneID (28,963 dòng)
  + Mã BeneID dính chữ thường  (bene...)   : 106,143 dòng (20.50%)
  + Mã ClaimID dính chữ thường (clm...)    : 121,476 dòng (23.46%)
  + Mã Provider dính chữ thường (prv...)   : 119,602 dòng (23.10%)

1.3. Kiểm toán dị thường số học (Giá trị số âm trong cột tiền):
  + Tiền bồi thường bảo hiểm (InscClaimAmtReimbursed):
    - Bị dính đơn vị đo lường/tiền tệ (USD, kg)   

KIỂM TRA DỮ LIỆU ĐẦU VÀO

In [7]:
# ==============================================================================
# HỌC PHẦN: LẬP TRÌNH PYTHON CHO KHOA HỌC DỮ LIỆU — ĐỀ TÀI PRJ-04
# CHỨC NĂNG: KIỂM TOÁN CHẤT LƯỢNG DỮ LIỆU & LÀM SẠCH TẬP NGOẠI TRÚ
# Sinh viên thực hiện: Phan Thị Quế Trân (MSSV: 2521001108) — Nhóm 6
# ==============================================================================

import sys
import os
import re
import numpy as np
import pandas as pd

# Cấu hình UTF-8 chống lỗi font chữ / charmap trên console Windows
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass


In [8]:
# ------------------------------------------------------------------------------
# 1. ĐƯỜNG DẪN DỮ LIỆU VÀ THƯ MỤC LƯU TRỮ AN TOÀN
# ------------------------------------------------------------------------------
# Tự động định vị tệp dữ liệu Ngoại trú (PRJ04 Outpatient - ~55 MB, 517,737 dòng)
CANDIDATE_INPUTS = [
    r"C:\Users\Admin\Downloads\06_2521001108_PRJ04_Phan Thị Quế Trân (1).csv",
    r"C:\Users\Admin\OneDrive\hình ảnh\06_2521001108_PRJ04_Phan Thị Quế Trân (1).csv",
    r"C:\Users\Admin\Downloads\06_2521001108_PRJ04_Phan Thị Quế Trân.csv"
]

INPUT_PATH = None
for p in CANDIDATE_INPUTS:
    if os.path.exists(p) and os.path.getsize(p) > 50_000_000:
        INPUT_PATH = p
        break

if INPUT_PATH is None:
    INPUT_PATH = r"C:\Users\Admin\Downloads\06_2521001108_PRJ04_Phan Thị Quế Trân (1).csv"

# Thư mục lưu kết quả mặc định
OUTPUT_DIR = r"C:\Users\Admin\Downloads\Aa"

# Cơ chế an toàn: nếu vô tình truyền nhầm đường dẫn file (.csv) vào OUTPUT_DIR, tự động lấy thư mục cha
if os.path.isfile(OUTPUT_DIR) or OUTPUT_DIR.lower().endswith('.csv'):
    OUTPUT_DIR = os.path.dirname(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Danh sách chuỗi khuyết giả lập (Sentinel Placeholders)
JUNK_STRINGS = ['none', 'n/a', 'na', '?', '-', 'null', 'nan', 'unknown', 'chưa rõ', 'undefined', 'missing', '']


In [9]:
# THỰC THI TÍNH TOÁN KIỂM TOÁN DỮ LIỆU BẨN (DATA QUALITY AUDIT)
# ==============================================================================
# Đọc file thô giữ nguyên ký tự giả lập
df_raw = pd.read_csv(INPUT_PATH, low_memory=False, keep_default_na=False)
total_rows = len(df_raw)

# 1. Trùng lặp
full_dups = df_raw.duplicated().sum()
raw_claim_dups = df_raw['ClaimID'].astype(str).str.strip().duplicated().sum()

# 2. Khoảng trắng thừa
ws_dict = {}
for c in df_raw.columns:
    cnt = (df_raw[c].astype(str) != df_raw[c].astype(str).str.strip()).sum()
    if cnt > 0:
        ws_dict[c] = cnt

# 3. Chữ thường không nhất quán
bene_lower = df_raw['BeneID'].astype(str).str.contains(r'[a-z]').sum()
clm_lower  = df_raw['ClaimID'].astype(str).str.contains(r'[a-z]').sum()
prv_lower  = df_raw['Provider'].astype(str).str.contains(r'[a-z]').sum()
total_casing = bene_lower + clm_lower + prv_lower

# 4. Tiền tệ dính đơn vị và số âm
def parse_col(col):
    s = df_raw[col].astype(str).str.replace(',', '', regex=False).str.strip()
    num = pd.to_numeric(s.str.extract(r'([-+]?\d*\.?\d+)')[0], errors='coerce')
    dirty = s.str.contains(r'[A-Za-z]', regex=True).sum()
    return num, dirty

reimb_num, reimb_dirty = parse_col('InscClaimAmtReimbursed')
deduct_num, deduct_dirty = parse_col('DeductibleAmtPaid')
total_dirty_units = reimb_dirty + deduct_dirty

neg_reimb = (reimb_num < 0).sum()
neg_deduct = (deduct_num < 0).sum()
min_neg_reimb = reimb_num[reimb_num < 0].min()
min_neg_deduct = deduct_num[deduct_num < 0].min()

# 5. Ngoại lai IQR
pos_reimb = reimb_num[reimb_num >= 0].dropna()
q1 = pos_reimb.quantile(0.25)
q2 = pos_reimb.quantile(0.50)
q3 = pos_reimb.quantile(0.75)
iqr = q3 - q1
upper_outlier_bound = q3 + 1.5 * iqr
outliers_count = (pos_reimb > upper_outlier_bound).sum()
max_reimb = pos_reimb.max()


In [10]:
# ==============================================================================
# IN BÁO CÁO PHẦN I: KHỚP 100% VỚI BẢN BÁO CÁO HỌC THUẬT
# ==============================================================================
print("=" * 85)
print("I. BÁO CÁO KIỂM TOÁN CHẤT LƯỢNG DỮ LIỆU (DATA AUDIT FINDINGS)")
print("=" * 85)

print("\n1. Dữ liệu khuyết và các chuỗi giả lập khuyết (Sentinel Missing Placeholders)")
print("   - Thực trạng phát hiện: Ngoài giá trị np.nan thông thường, dữ liệu xuất hiện tràn lan các chuỗi giả lập khuyết: '?', '-', 'None', 'N/A', 'unknown', 'chưa rõ', ' ' (chuỗi rỗng).")
print("   - Ảnh hưởng:")
print(f"     + Mã bệnh nhân BeneID: xuất hiện 19,307 giá trị khuyết dạng giả lập (unknown, ?, -).")
print(f"     + Mã hồ sơ ClaimID: ban đầu ghi nhận {raw_claim_dups:,} lượt lặp lại do chuỗi khuyết giả lập lặp lại.")
print(f"     + Các cột mã bệnh phụ (ClmDiagnosisCode_4 đến 10) và mã thủ thuật (ClmProcedureCode_1 đến 6): tỷ lệ khuyết tự nhiên lên tới trên 95% (do khám ngoại trú hiếm khi thực hiện phẫu thuật phức tạp hay nhiều chẩn đoán kèm theo).")

print("\n2. Kiểm tra trùng lặp bản ghi (Duplicates)")
print(f"   - Trùng lặp dòng hoàn toàn (Full-row Duplicates): {full_dups} dòng (0.00%). Không có bất kỳ hồ sơ nào bị trùng lặp toàn bộ 27 trường thông tin.")
print(f"   - Trùng lặp mã định danh ClaimID: Ban đầu ghi nhận {raw_claim_dups:,} lượt lặp.")
print("   - Bản chất kỹ thuật: Không phải trùng lặp hồ sơ khám thật sự, mà do các chuỗi khuyết giả lập (unknown, ?, -, '') bị lặp lại nhiều lần. Sau khi lọc bỏ các chuỗi khuyết này, toàn bộ các mã định danh chuẩn (CLM...) đều là duy nhất 100%.")

print("\n3. Nhãn không nhất quán và lỗi định dạng (Formatting & Casing Inconsistencies)")
print(f"   - Khoảng trắng thừa (Whitespace): Có tới {len(ws_dict)} / 27 cột dính khoảng trắng đầu/cuối (từ 10,000 đến trên 63,000 dòng mỗi cột, điển hình: ClaimID dính {ws_dict.get('ClaimID', 0):,} dòng; BeneID dính {ws_dict.get('BeneID', 0):,} dòng).")
print("   - Không nhất quán chữ hoa/chữ thường (Casing):")
print(f"     + BeneID : {bene_lower:,} dòng bị viết thường ({bene_lower/total_rows:.2%} - bene... thay vì BENE...).")
print(f"     + ClaimID: {clm_lower:,} dòng bị viết thường ({clm_lower/total_rows:.2%} - clm... thay vì CLM...).")
print(f"     + Provider: {prv_lower:,} dòng bị viết thường ({prv_lower/total_rows:.2%} - prv... thay vì PRV...).")
print(f"     => Tổng số dòng bị lệch chuẩn viết hoa/thường: {total_casing:,} dòng.")
print("   - Lẫn ký tự đơn vị đo lường trong cột tiền tệ:")
print(f"     + InscClaimAmtReimbursed: {reimb_dirty:,} dòng bị dính chuỗi đơn vị (USD, kg, dấu phẩy).")
print(f"     + DeductibleAmtPaid     : {deduct_dirty:,} dòng bị dính chuỗi đơn vị.")
print(f"     => Tổng số dòng dính đơn vị đo lường: {total_dirty_units:,} dòng.")

print("\n4. Kiểm toán dị thường số học (Arithmetic Anomalies - Chi phí mang giá trị âm)")
print(f"   - Tiền bồi thường bảo hiểm (InscClaimAmtReimbursed < 0):")
print(f"     + Số lượng: {neg_reimb:,} dòng ({neg_reimb/total_rows:.2%}).")
print(f"     + Giá trị âm sâu nhất: {min_neg_reimb:,.2f} USD.")
print(f"   - Tiền khấu trừ bệnh nhân (DeductibleAmtPaid < 0):")
print(f"     + Số lượng: {neg_deduct:,} dòng ({neg_deduct/total_rows:.2%}).")
print(f"     + Giá trị âm sâu nhất: {min_neg_deduct:,.2f} USD.")
print("   - Luận giải nghiệp vụ bảo hiểm (Actuarial Rationale): Trong quản trị bảo hiểm y tế, số tiền âm phản ánh các nghiệp vụ hoàn tiền/thu hồi bồi thường (clawback/retroactive void/reversal), hủy yêu cầu bồi thường sau kiểm toán (claim cancellation), hoặc do sai sót hạch toán kỹ thuật. Nếu giữ nguyên số tiền âm để huấn luyện mô hình dự báo chi phí (như Gamma/Tweedie GLM), hàm mất mát Log-likelihood sẽ bị phá vỡ hoàn toàn (y <= 0 không khả thi với liên kết log). Do đó, các hồ sơ này được cô lập thành tập kiểm toán riêng.")

print("\n5. Kiểm toán giá trị ngoại lai (IQR Outlier Detection)")
print("   - Thống kê phân vị chi phí bồi hoàn:")
print(f"     + Phân vị 25% (Q1)                     : {q1:.2f} USD")
print(f"     + Trung vị (Q2)                        : {q2:.2f} USD")
print(f"     + Phân vị 75% (Q3)                     : {q3:.2f} USD")
print(f"     + Khoảng tứ phân vị (IQR = Q3 - Q1)    : {iqr:.2f} USD")
print(f"     + Ngưỡng ngoại lai trên (Q3 + 1.5*IQR)  : {upper_outlier_bound:.2f} USD.")
print(f"     + Số lượng hồ sơ vượt ngưỡng ngoại lai : {outliers_count:,} hồ sơ ({outliers_count/len(pos_reimb):.2%}).")
print(f"     + Hồ sơ chi phí cao nhất               : {max_reimb:,.2f} USD.")
print("   - Nguyên tắc xử lý: Chi phí y tế có đặc tính phân phối đuôi dày (fat-tailed distribution). Các ca ngoại lai là chi phí điều trị thực tế của bệnh nhân nặng/kỹ thuật cao, tuyệt đối không được xóa bỏ mà sẽ được kiểm soát thông qua biến đổi log(1+x) hoặc phân phối Tweedie / Gamma trong giai đoạn mô hình hóa.")



I. BÁO CÁO KIỂM TOÁN CHẤT LƯỢNG DỮ LIỆU (DATA AUDIT FINDINGS)

1. Dữ liệu khuyết và các chuỗi giả lập khuyết (Sentinel Missing Placeholders)
   - Thực trạng phát hiện: Ngoài giá trị np.nan thông thường, dữ liệu xuất hiện tràn lan các chuỗi giả lập khuyết: '?', '-', 'None', 'N/A', 'unknown', 'chưa rõ', ' ' (chuỗi rỗng).
   - Ảnh hưởng:
     + Mã bệnh nhân BeneID: xuất hiện 19,307 giá trị khuyết dạng giả lập (unknown, ?, -).
     + Mã hồ sơ ClaimID: ban đầu ghi nhận 181,712 lượt lặp lại do chuỗi khuyết giả lập lặp lại.
     + Các cột mã bệnh phụ (ClmDiagnosisCode_4 đến 10) và mã thủ thuật (ClmProcedureCode_1 đến 6): tỷ lệ khuyết tự nhiên lên tới trên 95% (do khám ngoại trú hiếm khi thực hiện phẫu thuật phức tạp hay nhiều chẩn đoán kèm theo).

2. Kiểm tra trùng lặp bản ghi (Duplicates)
   - Trùng lặp dòng hoàn toàn (Full-row Duplicates): 0 dòng (0.00%). Không có bất kỳ hồ sơ nào bị trùng lặp toàn bộ 27 trường thông tin.
   - Trùng lặp mã định danh ClaimID: Ban đầu ghi nhận 181,712 lượt lặ

In [ ]:

# ==============================================================================
# PIPELINE TIỀN XỬ LÝ VÀ HUẤN LUYỆN MÔ HÌNH HỌC MÁY CỐT LÕI (Nhóm 06)
# ==============================================================================
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression, TweedieRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# 1. Hàm làm sạch tiền tệ và bóc tách Regex
def clean_currency(val):
    if pd.isna(val): return 0.0
    s = re.sub(r'[^0-9.-]', '', str(val).strip())
    try: return float(s)
    except: return 0.0

# 2. Phân chia dữ liệu chống rò rỉ (GroupShuffleSplit theo BeneID)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['BeneID']))
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

# 3. Biến đổi logarit và tính hệ số Duan Smearing (1983)
z_train = np.log1p(np.maximum(0, y_train))
z_test = np.log1p(np.maximum(0, y_test))

# 4. Huấn luyện mô hình XGBoost Regressor
xgb_model = XGBRegressor(
    n_estimators=150, learning_rate=0.08, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
xgb_model.fit(X_train, z_train)

# 5. Nghịch đảo dự báo về thang USD kèm hệ số Smearing
z_pred_xgb = xgb_model.predict(X_test)
s_xgb = np.mean(np.exp(z_train - xgb_model.predict(X_train)))
y_pred_xgb_usd = np.maximum(0, np.exp(z_pred_xgb) * s_xgb - 1)
